In [279]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import json, os, joblib, math, ast, random
from itertools import product

In [280]:
df = []

for path in os.listdir('Backup/System/'):
    index = int(path.split('_')[-1])
    
    with open(f'Backup/System/{path}/info.json', 'r', encoding='utf-8') as file:
        data = json.loads(file.read())
    
    name = data['model']
    data = data['info']
    data['name'] = name
    data['id'] = index
    df.append(data)


df = pd.DataFrame(df)
df

,r2,r2_adj,rmse,mae,name,id
0,0.838213,0.833603,69.445413,55.042335,linear_regression,2
1,0.947872,0.946539,39.143879,22.361166,linear_regression,5
2,0.838213,0.833603,69.445413,55.042335,linear_regression,1
3,0.838213,0.833603,69.445413,55.042335,linear_regression,3
4,1.000000,1.000000,0.000000,0.000000,linear_regression,4


In [281]:
df = df.sort_values(by='r2_adj', ascending=False)
df

,r2,r2_adj,rmse,mae,name,id
4,1.000000,1.000000,0.000000,0.000000,linear_regression,4
1,0.947872,0.946539,39.143879,22.361166,linear_regression,5
0,0.838213,0.833603,69.445413,55.042335,linear_regression,2
2,0.838213,0.833603,69.445413,55.042335,linear_regression,1
3,0.838213,0.833603,69.445413,55.042335,linear_regression,3


# SELECIONANDO MODELO

In [282]:
#target = df.loc[df.name == 'linear_regression'].sort_values(by='r2_adj',ascending=False)
target = df.sort_values(by='id', ascending=False)
target

,r2,r2_adj,rmse,mae,name,id
1,0.947872,0.946539,39.143879,22.361166,linear_regression,5
4,1.000000,1.000000,0.000000,0.000000,linear_regression,4
3,0.838213,0.833603,69.445413,55.042335,linear_regression,3
0,0.838213,0.833603,69.445413,55.042335,linear_regression,2
2,0.838213,0.833603,69.445413,55.042335,linear_regression,1


In [283]:
def loadModel(id):
    with open(f'Backup/System/model_{id}/info.json', 'r', encoding='utf-8') as file:
        data = json.loads(file.read())

    data['model'] = joblib.load(f'Backup/System/model_{id}/model.pkl')
    return data


id   = target.iloc[0].id
data = loadModel(id)
data

{'model': Pipeline(steps=[('scaler', StandardScaler()), ('model', LinearRegression())]),
 'params': {'memory': 'None',
  'steps': "[('scaler', StandardScaler()), ('model', LinearRegression())]",
  'transform_input': 'None',
  'verbose': 'False',
  'scaler': 'StandardScaler()',
  'model': 'LinearRegression()',
  'scaler__copy': 'True',
  'scaler__with_mean': 'True',
  'scaler__with_std': 'True',
  'model__copy_X': 'True',
  'model__fit_intercept': 'True',
  'model__n_jobs': 'None',
  'model__positive': 'False',
  'model__tol': '1e-06'},
 'K_CV': 5,
 'info': {'r2': 0.9478719104,
  'r2_adj': 0.9465390899,
  'rmse': 39.1438785081,
  'mae': 22.3611655457},
 'variables': ['u',
  'u(n-1)',
  'u(n-2)',
  'u(n-3)',
  'u(n-4)',
  'x(n-1)',
  'x(n-2)',
  'x(n-3)',
  'x(n-4)']}

In [284]:
variables = data['variables']
model = data['model']
model

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None


# CARREGANDO DADOS

In [285]:
df = pd.read_csv('../Dataset/Model.csv')
df.head(3)

,u,u(n-1),u(n-2),u(n-3),u(n-4),x(n-1),x(n-2),x(n-3),x(n-4),sensor
0,255,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1008
1,255,255.0,0.0,0.0,0.0,1008.0,0.0,0.0,0.0,699
2,255,255.0,255.0,0.0,0.0,699.0,1008.0,0.0,0.0,495


In [286]:
N_STATES = int(len(df.columns) / 2)
N_STATES

5

In [287]:
class StatesUpdater:
    buffer  = None
    initial = 0
    size    = 0

    def __init__(self, size, initial=0.00):
        self.initial = initial
        self.size    = size
        self.buffer  = np.array([initial for i in range(size)])
        
    def update(self, value):
        for i in range(self.size-1, 0, -1):
            self.buffer[i] = self.buffer[i-1]

        self.buffer[0] = value
        return self.buffer
    
    def get(self, var='x'):
        data = {}
        data[var] = self.buffer[0]
        
        for i in range(1, self.size):
            data[f'{var}(n-{i})'] = self.buffer[i]

        return {key: float(val) for key, val in data.items()}


states = StatesUpdater(3)
for i in range(1, 5): states.update(i); print(states.get())

{'x': 1.0, 'x(n-1)': 0.0, 'x(n-2)': 0.0}
{'x': 2.0, 'x(n-1)': 1.0, 'x(n-2)': 0.0}
{'x': 3.0, 'x(n-1)': 2.0, 'x(n-2)': 1.0}
{'x': 4.0, 'x(n-1)': 3.0, 'x(n-2)': 2.0}


In [288]:
model.feature_names_in_

array(['u', 'u(n-1)', 'u(n-2)', 'u(n-3)', 'u(n-4)', 'x(n-1)', 'x(n-2)',
       'x(n-3)', 'x(n-4)'], dtype=object)

In [289]:
class System:
    def __init__(self, model):
        self.inputs  = StatesUpdater(N_STATES) 
        self.outputs = StatesUpdater(N_STATES)
        self.model = model

    def update(self, value):
        self.outputs.update(value)
        data = self.get()
        del data['x']
        x    = self.model.predict(pd.DataFrame([data]))[0]
        self.inputs.update(x)
    
    def get(self):
        data = {}
        data.update(self.outputs.get('u'))
        data.update(self.inputs.get('x'))
        return data

In [290]:
system = System(model)
data   = []
last = 0    # último valor de saída 
tol  = 100  # tolerancia de salto de saídas

for i in range(100000):
    if i == 0:
        output = random.randint(0, 255)
    else:
        lower = max(0, last-tol)
        upper = min(255, last+tol)
        output = random.randint(lower, upper)
    
    last = output
    system.update(output)
    row = system.get()
    row['setpoint'] = row.pop('x') 
    data.append(row)


simulation = pd.DataFrame(data)
simulation

,u,u(n-1),u(n-2),u(n-3),u(n-4),x(n-1),x(n-2),x(n-3),x(n-4),setpoint
0,6.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,145.790744
1,31.0,6.0,0.0,0.0,0.0,145.790744,0.000000,0.000000,0.000000,147.154407
2,107.0,31.0,6.0,0.0,0.0,147.154407,145.790744,0.000000,0.000000,229.596953
3,31.0,107.0,31.0,6.0,0.0,229.596953,147.154407,145.790744,0.000000,175.944446
4,84.0,31.0,107.0,31.0,6.0,175.944446,229.596953,147.154407,145.790744,197.054728
...,...,...,...,...,...,...,...,...,...,...
99995,88.0,186.0,148.0,210.0,201.0,485.971381,412.545559,546.984866,467.847733,500.991633
99996,93.0,88.0,186.0,148.0,210.0,500.991633,485.971381,412.545559,546.984866,468.765581
99997,105.0,93.0,88.0,186.0,148.0,468.765581,500.991633,485.971381,412.545559,631.629116
99998,197.0,105.0,93.0,88.0,186.0,631.629116,468.765581,500.991633,485.971381,554.568988


In [291]:
simulation.to_csv('../Dataset/Simulation.csv', index=None)